# Notebook 5 — Feature engineering

**Job of this notebook:** build the features decided on in Notebook 4,
fitting every transformation on **train only** and applying it to val/test
— never re-fitting on new data.

**Reads:** `data/processed/{train,val,test}.parquet`.
**Writes:**
- `data/processed/features_{train,val,test}.parquet`
- `data/models/fitted_transformers.joblib` — every fitted encoder/scaler/imputer
- `data/models/feature_list.json` — the final feature column names

**Leakage rule:** only use information available *at prediction time* —
i.e. at the moment an order is placed and approved, before it ships.
Anything that only exists after delivery (delivered dates, review score,
delivery delay itself) is dropped before it ever reaches the model.


In [ ]:
import sys
sys.path.append("../src")

import json
import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from config import (
    TRAIN_PATH, VAL_PATH, TEST_PATH,
    FEATURES_TRAIN_PATH, FEATURES_VAL_PATH, FEATURES_TEST_PATH,
    TRANSFORMERS_PATH, FEATURE_LIST_PATH, LABEL_COL,
)

train = pd.read_parquet(TRAIN_PATH)
val = pd.read_parquet(VAL_PATH)
test = pd.read_parquet(TEST_PATH)

for df in (train, val, test):
    for c in df.columns:
        if "date" in c.lower() or "timestamp" in c.lower():
            df[c] = pd.to_datetime(df[c])

train.shape, val.shape, test.shape


## 1. Columns that must be dropped — they leak the future

These only exist *after* the order has already been delivered, so a
production pipeline predicting lateness at order time would never have
them. Keeping them would make the model look great in this notebook and
useless in production.


In [ ]:
LEAKY_COLS = [
    "order_delivered_customer_date",   # defines the label directly
    "delivery_delay_days",             # derived from the label
    "order_delivered_carrier_date",    # only known once the courier has picked up
    "review_score",                    # created after delivery
    "order_status",                    # 'delivered' is definitionally true for every labeled row
]

ID_COLS = ["order_id", "customer_id", "customer_unique_id", "product_id", "seller_id"]

def build_raw_features(df):
    df = df.copy()
    df["purchase_dow"] = df["order_purchase_timestamp"].dt.dayofweek
    df["purchase_month"] = df["order_purchase_timestamp"].dt.month
    df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour
    df["estimated_delivery_window_days"] = (
        df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]
    ).dt.total_seconds() / 86400
    df["approval_lag_hours"] = (
        df["order_approved_at"] - df["order_purchase_timestamp"]
    ).dt.total_seconds() / 3600
    df["cross_state"] = (df["customer_state"] != df["seller_state"]).astype(int)
    return df

train_f = build_raw_features(train)
val_f = build_raw_features(val)
test_f = build_raw_features(test)

DROP_COLS = LEAKY_COLS + ID_COLS + [
    "order_purchase_timestamp", "order_approved_at",
    "order_estimated_delivery_date", "max_shipping_limit_date",
]


## 2. Choose numerical and categorical feature columns

In [ ]:
candidate_cols = [c for c in train_f.columns if c not in DROP_COLS + ["is_late"]]

numeric_features = train_f[candidate_cols].select_dtypes(include=[np.number]).columns.tolist()
categorical_features = train_f[candidate_cols].select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


## 3. Fit transformers on train only, apply to val/test

Every fitted object (imputer, scaler, encoder) is saved — not just the
resulting table — because the production pipeline needs to load these
exact objects and transform new data with them, never re-fit.


In [ ]:
num_imputer = SimpleImputer(strategy="median")
num_imputer.fit(train_f[numeric_features])

scaler = StandardScaler()
scaler.fit(num_imputer.transform(train_f[numeric_features]))

cat_imputer = SimpleImputer(strategy="most_frequent")
cat_imputer.fit(train_f[categorical_features])

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=0.01)
ohe.fit(cat_imputer.transform(train_f[categorical_features]))

def transform(df):
    num = scaler.transform(num_imputer.transform(df[numeric_features]))
    num_df = pd.DataFrame(num, columns=numeric_features, index=df.index)

    cat = ohe.transform(cat_imputer.transform(df[categorical_features]))
    cat_df = pd.DataFrame(cat, columns=ohe.get_feature_names_out(categorical_features), index=df.index)

    out = pd.concat([num_df, cat_df], axis=1)
    out["is_late"] = df["is_late"].values
    return out

features_train = transform(train_f)
features_val = transform(val_f)
features_test = transform(test_f)

features_train.shape, features_val.shape, features_test.shape


In [ ]:
features_train.head()


## Artifacts

In [ ]:
features_train.to_parquet(FEATURES_TRAIN_PATH, index=False)
features_val.to_parquet(FEATURES_VAL_PATH, index=False)
features_test.to_parquet(FEATURES_TEST_PATH, index=False)

joblib.dump(
    {
        "num_imputer": num_imputer,
        "scaler": scaler,
        "cat_imputer": cat_imputer,
        "ohe": ohe,
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
    },
    TRANSFORMERS_PATH,
)

feature_list = [c for c in features_train.columns if c != "is_late"]
with open(FEATURE_LIST_PATH, "w") as f:
    json.dump(feature_list, f, indent=2)

print(f"Saved {len(feature_list)} features.")
print("->", FEATURES_TRAIN_PATH)
print("->", TRANSFORMERS_PATH)
print("->", FEATURE_LIST_PATH)


**In production:** the pipeline loads `fitted_transformers.joblib` and
calls `.transform(...)` on new orders with the exact same objects fitted
here — it never calls `.fit(...)` again on live data.
